## As seen in the EDA, there is a massive 7 month gap for plant 710. I will use Donor Imputation Method to fill the missing values based on plants 515, which possess a similar range of `m3` per hour (rounded, e.g. 9:00 for all remissions between 9:00 and 9:59) and similar count of  remissions (rows) also per hour.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor

# Config
DATA_PATH = "../data/processed/remissions_db_cleaned.xlsx"
PLANT_TARGET = "710"
PLANT_PRIMARY = "515"
GAP_START = "2023-05-17"
GAP_END = "2024-02-02"
FALLBACK_TOP_K = 3
RANDOM_STATE = 42
# Optional: force weekend schedule for 710 (set to None to disable)
DOW_SCALE_OVERRIDE = {5: 0.7, 6: 0.2}
# Blend dow scaling toward 1.0 to avoid over-suppressing counts
DOW_BLEND = 0.6
# Use 710 vs 515 ratios for weekend scaling
USE_515_DOW_RATIOS = True
# Optional: scale per-row volume by 710 vs 515 dow ratios
APPLY_VOLUME_DOW_SCALE = False
VOLUME_DOW_BLEND = 0.5
VOLUME_DOW_CLIP = (0.6, 1.4)
# Keep real 515 volume distribution (preserves max values)
USE_515_FOR_VOLUME = True
# Exclude u_Volumen from RF so it keeps real distribution
EXCLUDE_VOLUME_FROM_RF = True
# Blend RF hourly counts toward 710 pre-gap mean by day-of-week and hour
COUNT_BLEND_WEEKDAY = 0.6
COUNT_BLEND_WEEKEND = 0.0
# Count sampling from 515 by (hour, dow, month), scaled by both ratios
USE_515_COUNT_SAMPLING = True
COUNT_SAMPLE_SCALE_HOUR = True
COUNT_SAMPLE_SCALE_DOW = True
COUNT_MAX_Q = 0.98

# Load and parse
remissions = pd.read_excel(DATA_PATH)
for col in ["order_date", "typed_time", "start_time", "at_plant_time"]:
    if col in remissions.columns:
        remissions[col] = pd.to_datetime(remissions[col], errors="coerce")

remissions["ship_plant_code"] = remissions["ship_plant_code"].astype(str).str.strip()

# Helpers
def make_time_features(dt_series):
    dt = pd.to_datetime(dt_series)
    return pd.DataFrame({
        "hour": dt.dt.hour,
        "day_of_week": dt.dt.dayofweek,
        "month": dt.dt.month,
        "day_of_year": dt.dt.dayofyear,
        "is_weekend": (dt.dt.dayofweek >= 5).astype(int),
    })

# Build donor pool based on similarity to plant 515 (pre-gap)
gap_start = pd.Timestamp(GAP_START)
ref = remissions.dropna(subset=["typed_time"]).copy()
ref = ref[ref["typed_time"] < gap_start]
ref["hour"] = ref["typed_time"].dt.hour

counts = ref.groupby(["ship_plant_code", "hour"]).size().unstack(fill_value=0)
vols = ref.groupby(["ship_plant_code", "hour"])["u_Volumen"].mean().unstack(fill_value=0)

def plant_profile(plant_code):
    if plant_code in counts.index:
        c = counts.loc[plant_code].reindex(range(24), fill_value=0)
    else:
        c = pd.Series(0, index=range(24))
    if plant_code in vols.index:
        v = vols.loc[plant_code].reindex(range(24), fill_value=0)
    else:
        v = pd.Series(0, index=range(24))
    if c.sum() > 0:
        c = c / c.sum()
    if v.sum() > 0:
        v = v / v.sum()
    return np.concatenate([c.values, v.values])

primary_profile = plant_profile(PLANT_PRIMARY)
sims = {}
for plant in counts.index:
    if plant in [PLANT_PRIMARY, PLANT_TARGET]:
        continue
    vec = plant_profile(plant)
    denom = np.linalg.norm(primary_profile) * np.linalg.norm(vec)
    sim = float(primary_profile.dot(vec) / denom) if denom > 0 else 0.0
    sims[plant] = sim

fallback_plants = [p for p, _ in sorted(sims.items(), key=lambda kv: kv[1], reverse=True)[:FALLBACK_TOP_K]]
donor_plants = [PLANT_PRIMARY] + fallback_plants
print("Donor plants:", donor_plants)

# Donor data
pool = remissions[remissions["ship_plant_code"].isin(donor_plants)].copy()
pool = pool.dropna(subset=["typed_time"])

# Pre-gap 710 baseline for scaling
p710_hist = remissions[
    (remissions["ship_plant_code"] == PLANT_TARGET) &
    (remissions["typed_time"].notna()) &
    (remissions["typed_time"] < gap_start)
]].copy()

# 515 pre-gap ratios for diagnostics and optional scaling
p515_hist = remissions[
    (remissions["ship_plant_code"] == PLANT_PRIMARY) &
    (remissions["typed_time"].notna()) &
    (remissions["typed_time"] < gap_start)
]].copy()
ratio_515_counts = None
ratio_515_volume = None
ratio_515_hour = None
if not p710_hist.empty and not p515_hist.empty:
    p710_hist["day_of_week"] = p710_hist["typed_time"].dt.dayofweek
    p515_hist["day_of_week"] = p515_hist["typed_time"].dt.dayofweek
    ratio_515_counts = (
        p710_hist.groupby("day_of_week").size() /
        p515_hist.groupby("day_of_week").size()
    ).replace([np.inf, -np.inf], np.nan).reindex(range(7))
    if "u_Volumen" in remissions.columns:
        ratio_515_volume = (
            p710_hist.groupby("day_of_week")["u_Volumen"].sum() /
            p515_hist.groupby("day_of_week")["u_Volumen"].sum()
        ).replace([np.inf, -np.inf], np.nan).reindex(range(7))
    ratio_515_hour = (
        p710_hist.groupby(p710_hist["typed_time"].dt.hour).size() /
        p515_hist.groupby(p515_hist["typed_time"].dt.hour).size()
    ).replace([np.inf, -np.inf], np.nan).reindex(range(24))
    ratio_table = pd.DataFrame({
        "ratio_count_710_vs_515": ratio_515_counts,
        "ratio_volume_710_vs_515": ratio_515_volume,
    })
    print("710 vs 515 day-of-week ratios (pre-gap):")
    print(ratio_table)
    ratio_515_counts = ratio_515_counts.fillna(1.0).clip(0.1, 3.0)
    if ratio_515_volume is not None:
        ratio_515_volume = ratio_515_volume.fillna(1.0).clip(0.3, 2.0)
    ratio_515_hour = ratio_515_hour.fillna(1.0).clip(0.3, 3.0)

p515_hourly_counts = None
idx_cnt_full = None
idx_cnt_hour = None
if not p515_hist.empty:
    p515_hourly = p515_hist.copy()
    p515_hourly["hour_bucket"] = p515_hourly["typed_time"].dt.floor("h")
    p515_hourly_counts = p515_hourly.groupby("hour_bucket").size().reset_index(name="count")
    p515_hourly_counts["hour"] = p515_hourly_counts["hour_bucket"].dt.hour
    p515_hourly_counts["day_of_week"] = p515_hourly_counts["hour_bucket"].dt.dayofweek
    p515_hourly_counts["month"] = p515_hourly_counts["hour_bucket"].dt.month
    idx_cnt_full = p515_hourly_counts.groupby(["hour", "day_of_week", "month"]).indices
    idx_cnt_hour = p515_hourly_counts.groupby(["hour"]).indices

pool["hour"] = pool["typed_time"].dt.hour
pool["day_of_week"] = pool["typed_time"].dt.dayofweek
donor_hour_counts = pool.groupby("hour").size()
donor_dow_counts = pool.groupby("day_of_week").size()

if not p710_hist.empty:
    p710_hist["hour"] = p710_hist["typed_time"].dt.hour
    p710_hist["day_of_week"] = p710_hist["typed_time"].dt.dayofweek
    ratio_by_hour = (p710_hist.groupby("hour").size() / donor_hour_counts).replace([np.inf, -np.inf], np.nan)
    ratio_by_hour = ratio_by_hour.reindex(range(24), fill_value=np.nan)
    ratio_mean = ratio_by_hour.mean()
    ratio_by_hour = ratio_by_hour.fillna(ratio_mean if pd.notna(ratio_mean) else 1.0)
    ratio_by_hour = ratio_by_hour.clip(0.3, 3.0)
    ratio_by_dow = (p710_hist.groupby("day_of_week").size() / donor_dow_counts).replace([np.inf, -np.inf], np.nan)
    ratio_by_dow = ratio_by_dow.reindex(range(7), fill_value=np.nan)
    ratio_dow_mean = ratio_by_dow.mean()
    ratio_by_dow = ratio_by_dow.fillna(ratio_dow_mean if pd.notna(ratio_dow_mean) else 1.0)
    ratio_by_dow = ratio_by_dow.clip(0.2, 3.0)
else:
    ratio_by_hour = pd.Series(1.0, index=range(24))
    ratio_by_dow = pd.Series(1.0, index=range(7))

if USE_515_DOW_RATIOS and ratio_515_counts is not None:
    for dow in [5, 6]:
        ratio_by_dow.loc[dow] = ratio_515_counts.loc[dow]

if DOW_SCALE_OVERRIDE:
    for dow, val in DOW_SCALE_OVERRIDE.items():
        ratio_by_dow.loc[dow] = val

ratio_by_dow = ratio_by_dow.clip(0.0, 3.0)
if DOW_BLEND is not None:
    ratio_by_dow = (1 - DOW_BLEND) + DOW_BLEND * ratio_by_dow

# Train hourly count model on donor pool
pool["hour_bucket"] = pool["typed_time"].dt.floor("h")
hourly_counts = pool.groupby("hour_bucket").size().reset_index(name="count")
X_counts = make_time_features(hourly_counts["hour_bucket"])
y_counts = hourly_counts["count"]

donor_hourly_counts = hourly_counts["count"]
max_count = int(np.ceil(np.quantile(donor_hourly_counts, COUNT_MAX_Q)))
if USE_515_COUNT_SAMPLING and p515_hourly_counts is not None:
    max_count = int(np.ceil(np.quantile(p515_hourly_counts["count"], COUNT_MAX_Q)))

rf_count = RandomForestRegressor(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_count.fit(X_counts, y_counts)

# Gap hours for plant 710
raw_gap_end = pd.Timestamp(GAP_END)
gap_end = raw_gap_end + pd.Timedelta(hours=23)
full_gap_hours = pd.date_range(gap_start, gap_end, freq="h")

p710 = remissions[
    (remissions["ship_plant_code"] == PLANT_TARGET) &
    (remissions["typed_time"].notna())
]
existing_hours = p710["typed_time"].dt.floor("h").unique()
gap_hours = full_gap_hours.difference(existing_hours)

rng = np.random.default_rng(RANDOM_STATE)
if USE_515_COUNT_SAMPLING and p515_hourly_counts is not None:
    base_counts = []
    for ts in gap_hours:
        key = (ts.hour, ts.dayofweek, ts.month)
        idxs = idx_cnt_full.get(key) if idx_cnt_full is not None else None
        if idxs is None:
            idxs = idx_cnt_hour.get(ts.hour) if idx_cnt_hour is not None else None
            if idxs is None:
                idxs = np.arange(len(p515_hourly_counts))
        idx = rng.choice(idxs)
        base_counts.append(p515_hourly_counts.iloc[idx]["count"])
    pred_counts = np.array(base_counts, dtype=float)

    if COUNT_SAMPLE_SCALE_HOUR and ratio_515_hour is not None:
        hour_scale_series = ratio_515_hour
    else:
        hour_scale_series = pd.Series(1.0, index=range(24))
    scale_hour = pd.Series(gap_hours.hour).map(hour_scale_series).fillna(1.0).to_numpy()

    if COUNT_SAMPLE_SCALE_DOW and ratio_515_counts is not None:
        dow_scale_series = ratio_515_counts.copy()
    else:
        dow_scale_series = pd.Series(1.0, index=range(7))
    if DOW_SCALE_OVERRIDE:
        for dow, val in DOW_SCALE_OVERRIDE.items():
            dow_scale_series.loc[dow] = val
    if DOW_BLEND is not None:
        dow_scale_series = (1 - DOW_BLEND) + DOW_BLEND * dow_scale_series
    scale_dow = pd.Series(gap_hours.dayofweek).map(dow_scale_series).fillna(1.0).to_numpy()

    pred_counts = np.clip(np.rint(pred_counts * scale_hour * scale_dow), 0, max_count).astype(int)
else:
    X_gap = make_time_features(pd.Series(gap_hours))
    pred_counts = rf_count.predict(X_gap)
    scale_hour = pd.Series(gap_hours.hour).map(ratio_by_hour).fillna(1.0).to_numpy()
    scale_dow = pd.Series(gap_hours.dayofweek).map(ratio_by_dow).fillna(1.0).to_numpy()
    pred_counts = np.clip(np.rint(pred_counts * scale_hour * scale_dow), 0, max_count).astype(int)

# Optional: calibrate toward 710 pre-gap mean by (day_of_week, hour)
if (COUNT_BLEND_WEEKDAY is not None or COUNT_BLEND_WEEKEND is not None) and not p710_hist.empty:
    p710_hourly = p710_hist.copy()
    p710_hourly["hour_bucket"] = p710_hourly["typed_time"].dt.floor("h")
    p710_hourly_counts = p710_hourly.groupby("hour_bucket").size().reset_index(name="count")
    p710_hourly_counts["day_of_week"] = p710_hourly_counts["hour_bucket"].dt.dayofweek
    p710_hourly_counts["hour"] = p710_hourly_counts["hour_bucket"].dt.hour
    mean_by_hd = p710_hourly_counts.groupby(["day_of_week", "hour"])["count"].mean()
    mean_by_hd = mean_by_hd.reindex(pd.MultiIndex.from_product([range(7), range(24)]))
    gap_index = pd.MultiIndex.from_arrays([gap_hours.dayofweek, gap_hours.hour])
    expected = mean_by_hd.reindex(gap_index).to_numpy()
    overall_mean = p710_hourly_counts["count"].mean()
    expected = np.where(np.isnan(expected), overall_mean, expected)
    weekday_blend = 0.0 if COUNT_BLEND_WEEKDAY is None else COUNT_BLEND_WEEKDAY
    weekend_blend = 0.0 if COUNT_BLEND_WEEKEND is None else COUNT_BLEND_WEEKEND
    is_weekend = pd.Series(gap_hours.dayofweek).isin([5, 6]).to_numpy()
    blend = np.where(is_weekend, weekend_blend, weekday_blend)
    pred_counts = np.rint((1 - blend) * pred_counts + blend * expected)
    pred_counts = np.clip(pred_counts, 0, max_count).astype(int)

# Sample donor rows to create synthetic remissions
pool["day_of_week"] = pool["typed_time"].dt.dayofweek
pool["month"] = pool["typed_time"].dt.month

idx_full = pool.groupby(["hour", "day_of_week", "month"]).indices
idx_hour = pool.groupby(["hour"]).indices

sampled_idx = []
for hour_ts, n in zip(gap_hours, pred_counts):
    if n == 0:
        continue
    key = (hour_ts.hour, hour_ts.dayofweek, hour_ts.month)
    idxs = idx_full.get(key)
    if idxs is None:
        idxs = idx_hour.get(hour_ts.hour)
        if idxs is None:
            idxs = np.arange(len(pool))
    idxs = np.asarray(idxs)
    choice = rng.choice(idxs, size=n, replace=True)
    sampled_idx.append(choice)

if not sampled_idx:
    raise ValueError("No synthetic rows created. Check donor data or model predictions.")

sampled_idx = np.concatenate(sampled_idx)
synthetic = pool.iloc[sampled_idx].copy().reset_index(drop=True)

# Target hours for synthetic rows
rep_hours = np.repeat(gap_hours.values, pred_counts)
rep_hours = pd.to_datetime(rep_hours)

# Optional: use real 515 volume distribution
if USE_515_FOR_VOLUME and "u_Volumen" in synthetic.columns:
    vol_pool = p515_hist.dropna(subset=["typed_time", "u_Volumen"]).copy()
    if not vol_pool.empty:
        vol_pool["hour"] = vol_pool["typed_time"].dt.hour
        vol_pool["day_of_week"] = vol_pool["typed_time"].dt.dayofweek
        vol_pool["month"] = vol_pool["typed_time"].dt.month
        vol_idx_full = vol_pool.groupby(["hour", "day_of_week", "month"]).indices
        vol_idx_hour = vol_pool.groupby(["hour"]).indices
        vol_vals = []
        for ts in rep_hours:
            key = (ts.hour, ts.dayofweek, ts.month)
            idxs = vol_idx_full.get(key)
            if idxs is None:
                idxs = vol_idx_hour.get(ts.hour)
                if idxs is None:
                    idxs = np.arange(len(vol_pool))
            idx = rng.choice(idxs)
            vol_vals.append(vol_pool.iloc[idx]["u_Volumen"])
        synthetic["u_Volumen"] = vol_vals

# Preserve minute/second offsets from donor rows
for col in ["typed_time", "start_time", "at_plant_time"]:
    if col in synthetic.columns:
        src = pd.to_datetime(synthetic[col], errors="coerce")
        offset = src - src.dt.floor("h")
        synthetic[col] = rep_hours + offset.fillna(pd.Timedelta(0))

synthetic["order_date"] = rep_hours.normalize()

# Keep donor plant as metadata, overwrite target plant
synthetic["imputed_source_plant"] = synthetic["ship_plant_code"]
synthetic["ship_plant_code"] = PLANT_TARGET
synthetic["is_imputed"] = True
synthetic["imputed_method"] = "donor+rf"
synthetic["imputed_range"] = f"{GAP_START} to {GAP_END}"

if "year" in synthetic.columns:
    synthetic["year"] = synthetic["order_date"].dt.year

# Clear identifier-like codes except plant code
for col in remissions.columns:
    if col != "ship_plant_code" and col.lower().endswith("_code"):
        synthetic[col] = pd.NA

# Impute numeric columns with RF using donor pool
numeric_cols = remissions.select_dtypes(include=["number"]).columns.tolist()
exclude = [c for c in numeric_cols if "code" in c.lower() or c.lower().endswith("_id")]
exclude.append("year")
numeric_cols = [c for c in numeric_cols if c not in exclude]
if EXCLUDE_VOLUME_FROM_RF and "u_Volumen" in numeric_cols:
    numeric_cols.remove("u_Volumen")

if numeric_cols:
    train_mask = pool[numeric_cols].notna().all(axis=1)
    train_mask &= pool["typed_time"].notna()

    X_train = make_time_features(pool.loc[train_mask, "typed_time"])
    X_train["plant_code"] = pool.loc[train_mask, "ship_plant_code"].astype(str)
    X_train = pd.get_dummies(X_train, columns=["plant_code"], drop_first=False)

    y_train = pool.loc[train_mask, numeric_cols]

    rf_num = RandomForestRegressor(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    model = MultiOutputRegressor(rf_num)
    model.fit(X_train, y_train)

    X_pred = make_time_features(synthetic["typed_time"])
    X_pred["plant_code"] = synthetic["imputed_source_plant"].astype(str)
    X_pred = pd.get_dummies(X_pred, columns=["plant_code"], drop_first=False)
    X_pred = X_pred.reindex(columns=X_train.columns, fill_value=0)

    int_cols = [c for c in numeric_cols if pd.api.types.is_integer_dtype(remissions[c])]
    synthetic[numeric_cols] = synthetic[numeric_cols].astype("Float64")

    y_pred = model.predict(X_pred)
    pred_df = pd.DataFrame(y_pred, columns=numeric_cols, index=synthetic.index)
    synthetic.loc[:, numeric_cols] = pred_df

    if APPLY_VOLUME_DOW_SCALE and ratio_515_volume is not None and "u_Volumen" in synthetic.columns:
        dow_factor = synthetic["typed_time"].dt.dayofweek.map(ratio_515_volume).fillna(1.0)
        dow_factor = dow_factor.clip(VOLUME_DOW_CLIP[0], VOLUME_DOW_CLIP[1])
        if VOLUME_DOW_BLEND is not None:
            dow_factor = (1 - VOLUME_DOW_BLEND) + VOLUME_DOW_BLEND * dow_factor
        synthetic["u_Volumen"] = synthetic["u_Volumen"] * dow_factor

    if not p710_hist.empty:
        q_low = p710_hist[numeric_cols].quantile(0.01)
        q_high = p710_hist[numeric_cols].quantile(0.99)
    else:
        q_low = pd.Series(index=numeric_cols, dtype=float)
        q_high = pd.Series(index=numeric_cols, dtype=float)

    donor_low = pool[numeric_cols].quantile(0.01)
    donor_high = pool[numeric_cols].quantile(0.99)
    q_low = q_low.fillna(donor_low)
    q_high = q_high.fillna(donor_high)

    for col in numeric_cols:
        synthetic[col] = synthetic[col].clip(q_low[col], q_high[col])

    if "u_Volumen" in synthetic.columns:
        synthetic["u_Volumen"] = synthetic["u_Volumen"].clip(lower=0.1)
    if "u_Cicle" in synthetic.columns:
        synthetic["u_Cicle"] = synthetic["u_Cicle"].clip(lower=1.0)

    if int_cols:
        synthetic[int_cols] = synthetic[int_cols].round().astype("Int64")

# Merge and export
imputed = pd.concat([remissions, synthetic], ignore_index=True)
imputed = imputed.sort_values("typed_time").reset_index(drop=True)

output_path = "../data/processed/remissions_db_imputed_710.xlsx"
imputed.to_excel(output_path, index=False)

print("Synthetic rows:", len(synthetic))
print("Total rows:", len(imputed))
print("Output:", output_path)

Donor plants: ['515', '512', '514', '511']
710 vs 515 day-of-week ratios (pre-gap):
             ratio_count_710_vs_515  ratio_volume_710_vs_515
day_of_week                                                 
0                          1.273359                 1.246449
1                          1.242764                 1.183465
2                          1.232114                 1.196929
3                          1.211494                 1.147802
4                          1.282900                 1.262361
5                          1.062741                 0.987581
6                          1.000000                 2.000000
Synthetic rows: 15313
Total rows: 353521
Output: ../data/processed/remissions_db_imputed_710.xlsx


In [2]:
print(type(sampled_idx), len(sampled_idx))
print(sampled_idx[:10])
print(pool.index[:10])

<class 'numpy.ndarray'> 43964
[111487 175993 175988 173806 173806 178663  11953 127295  11867  11832]
Index([2781, 2782, 2783, 2784, 2785, 2786, 2787, 2788, 2789, 2790], dtype='int64')
